# STL-Rocket: Explanations of Time-Series Classifiers

## Setup

In [35]:
import sys
sys.path.insert(0, "..")  # repo root, for the local (non pip-installed) stlrocket package

import copy
from collections import defaultdict
from functools import reduce
import numpy as np
import pandas as pd
from torcheck.stl import Atom, Not, And, Or

from stlrocket.data import load_dataset
from stlrocket.config import ExperimentConfig
from stlrocket.features import build_formula_bank, eval_robustness, shift_atom_thresholds
from stlrocket.classifier import train_classifier, evaluate_classifier
from stlrocket.evaluation import evaluate_global
from stlrocket.simplification import simplify_global, simplify_data_aware, round_thresholds

In [58]:
config = ExperimentConfig(
    # Dataset
    dataset = "HandMovementDirection",
    n_formulas = 10000,
    depth_max = 2,
    until_weight = 0.0,

    # Classifier
    cv = 3,

    # Explanation
    pool_size = 10,
    precision_threshold = 0.9,

    # Global explanation simplification
    simplify_agreement = 0.95,
    simplify_min_gain = 0.0,
    simplify_decimals = 1,

    # Experiment loop
    n_run = 10,
    base_seed = 0, # run i uses seed=base_seed+i

    # Explanations
    explain = True,

    # Output
    output_dir = "results",

    # Hardware
    device = "cpu"
)

# Beam search hyper-parameters (Section 3.3.2)
ALPHA = config.precision_threshold   # precision level α
BEAM_WIDTH = 5                       # B
N_SAMPLE = 10                        # m, top-IG formulae proposed per state
N_MAX = 3                           # n_max, max number of conjuncts

## Data, features and classifier

In [59]:
X_tr_raw, y_tr, X_te_raw, y_te = load_dataset(config.dataset)
print(f"train: {X_tr_raw.shape}, test: {X_te_raw.shape}, classes: {np.unique(y_tr)}")

formulas, X_tr_feats, X_te_feats = build_formula_bank(X_tr_raw, X_te_raw, config, config.base_seed)
model = train_classifier(X_tr_feats, y_tr, config)
W, b = model.coef_, model.intercept_

print(f"balanced accuracy: train {evaluate_classifier(model, X_tr_feats, y_tr)['balanced_accuracy']:.3f}, "
      f"test {evaluate_classifier(model, X_te_feats, y_te)['balanced_accuracy']:.3f}")
print(f"non-zero columns of W: {(W != 0).any(axis=0).sum()} / {W.shape[1]}")

train: (160, 10, 400), test: (74, 10, 400), classes: ['backward' 'forward' 'left' 'right']
balanced accuracy: train 0.713, test 0.513
non-zero columns of W: 36 / 10000


## Explanation method

**Attribution (eq. 18–20).** Integrated gradients with baseline $\hat{\mathbf x}'=\mathbf 0$ (train-mean robustness). For binary problems glmnet returns a single row $w$, i.e. the logits $(0,\, w^\top\hat{\mathbf x})$. The candidate formulae are $J=\{j : \mathrm{IG}_j>0\}$, with orientation $\epsilon_j=\mathrm{sign}(d_j)$.

**Reparametrization (eq. 21).** With the cut $c=\epsilon_j\mu_j+s\,\sigma_j$, the robustness of $\tilde\phi_{j,c}$ is $\sigma_j(u_j-s)$, where $u_j=\epsilon_j\hat x_j$ is the oriented standardized feature. So $\tilde\phi_{j,c}$ covers exactly $\{\tau' : u_j(\tau')>s\}$: the search runs on standardized train features, and STL formulas are built only for the final explanation.

**Beam search.** A local explanation is a conjunction of at most $n_{\max}$ cuts with $s\in[0,u_j(\tau))$, so $\tau$ always satisfies it. It maximizes the number of covered train series with $f=k$ subject to precision $>\alpha$ (w.r.t. the model predictions $f$). Each state is expanded with its $m$ unused formulae of largest IG; each cut sits at the midpoint of a gap between train series. If no rule is feasible, the highest-precision visited state is returned.

In [60]:
K = len(model.classes_)
W_full = np.vstack([np.zeros_like(W), W]) if W.shape[0] == 1 else W   # (K, M)
ALPHAS = np.linspace(0.0, 1.0, 1001)


def softmax(z):
    e = np.exp(z - z.max(axis=-1, keepdims=True))
    return e / e.sum(axis=-1, keepdims=True)


def integrated_gradients(x_hat):
    # Eq. (18): IG_j = d_j x̂_j, d_j = w_kj - sum_l p̄_l w_lj
    z = W_full @ x_hat
    k = int(np.argmax(z))
    P = softmax(ALPHAS[:, None] * z[None, :])      # (steps, K)
    p_bar = np.trapezoid(P, ALPHAS, axis=0)        # (K,)
    d = W_full[k] - p_bar @ W_full                 # (M,)
    return k, d * x_hat, d, z


# Model predictions f(τ) as class indices (rows of W_full / model.classes_)
f_tr = np.argmax(X_tr_feats @ W_full.T, axis=1)
f_te = np.argmax(X_te_feats @ W_full.T, axis=1)
assert (model.classes_[f_te] == model.predict(X_te_feats)).all()

# Completeness, eq. (19)
k, ig, d, z = integrated_gradients(X_te_feats[0])
assert np.isclose(ig.sum(), np.log(softmax(z)[k]) + np.log(K), atol=1e-3)

In [61]:
def candidate_cuts(u, u_tau, cov, pos, alpha):
    # Cuts s ∈ [0, u_j(τ)), restricted to the currently covered series (the others give
    # the same covered sets). Each distinct covered set {u > s} is represented by the
    # midpoint of its gap (max-margin cut), clamped to [0, u_j(τ)).
    # Returns (s_G, s_F): max gain (ties -> coverage) and, among feasible cuts, the one with
    # most true positives (ties -> highest precision), or None.
    vals, lab = u[cov], pos[cov]
    all_sorted, pos_sorted = np.sort(vals), np.sort(vals[lab])
    lo = np.unique(np.concatenate([[0.0], vals[(vals >= 0) & (vals < u_tau)]]))  # ascending
    idx = np.searchsorted(all_sorted, lo, side="right")
    n_cov = len(all_sorted) - idx
    n_pos = len(pos_sorted) - np.searchsorted(pos_sorted, lo, side="right")
    nxt = np.append(all_sorted, np.inf)[idx]                         # smallest covered value above lo
    cands = (lo + np.minimum(nxt, u_tau)) / 2                        # same covered set as lo
    gain = n_pos - alpha * n_cov                                     # eq. (23)
    s_G = cands[np.lexsort((-n_cov, -gain))[0]]
    feasible = np.flatnonzero((n_cov > 0) & (n_pos > alpha * n_cov))
    s_F = cands[feasible[np.lexsort((n_cov[feasible], -n_pos[feasible]))[0]]] if len(feasible) else None
    return s_G, s_F


def explain_local(x_hat, alpha=ALPHA, beam_width=BEAM_WIDTH, m=N_SAMPLE, n_max=N_MAX):
    # Attribution-guided beam search. Conjuncts are (j, ε_j, s) with s in standardized units.
    # Deterministic: each state is expanded with its m unused formulae of largest IG.
    k, ig, d, _ = integrated_gradients(x_hat)
    J = np.flatnonzero(ig > 0)                     # eq. (20)
    if len(J) == 0:
        return dict(k=k, conjuncts=[], feasible=False)
    eps = np.sign(d[J])
    U = X_tr_feats[:, J] * eps                     # u_j(τ^(i)), (N, |J|)
    u_tau = x_hat[J] * eps                         # u_j(τ) > 0 for j ∈ J
    pos = f_tr == k

    beam = [((), np.ones(len(pos), dtype=bool))]   # B_0 = {⊤}
    best, best_key = None, (0, 0.0)                # L*, (true positives, precision) of L*
    fallback, fallback_prec = None, -1.0           # highest-precision visited state
    for _ in range(n_max):
        expansions = {}
        for conj, cov in beam:
            used = [p for p, _ in conj]
            avail = np.setdiff1d(np.arange(len(J)), used)
            if len(avail) == 0:
                continue
            # Deterministic: the m unused formulae with the largest IG
            picks = avail[np.argsort(-ig[J][avail], kind="stable")[:m]]
            for p in picks:
                s_G, s_F = candidate_cuts(U[:, p], u_tau[p], cov, pos, alpha)
                if s_F is not None:
                    cov_F = cov & (U[:, p] > s_F)
                    n_tp_F = int((cov_F & pos).sum())
                    key_F = (n_tp_F, n_tp_F / int(cov_F.sum()))
                    if key_F > best_key:
                        best, best_key = (conj + ((p, s_F),), cov_F), key_F
                cov_G = cov & (U[:, p] > s_G)
                n, n_pos = int(cov_G.sum()), int((cov_G & pos).sum())
                if n > 0 and n_pos / n > fallback_prec:
                    fallback, fallback_prec = (conj + ((p, s_G),), cov_G), n_pos / n
                score = (n_pos - alpha * n, n)
                key = frozenset(used + [p])        # merge states with the same conjuncts
                if key not in expansions or score > expansions[key][0]:
                    expansions[key] = (score, conj + ((p, s_G),), cov_G, n_pos)
        # prune: true positives cannot grow, so states that cannot beat L* are useless
        survivors = sorted((e for e in expansions.values()
                            if e[3] > best_key[0] or (e[3] == best_key[0] and best_key[1] < 1.0)),
                           key=lambda e: e[0], reverse=True)
        beam = [(c, cv) for _, c, cv, _ in survivors[:beam_width]]
        if not beam:
            break

    chosen = best if best is not None else fallback
    conjuncts = [] if chosen is None else [(int(J[p]), float(eps[p]), float(s)) for p, s in chosen[0]]
    return dict(k=k, conjuncts=conjuncts, feasible=best is not None)


def covered(conjuncts, X_feats):
    # Feature-space evaluation of ρ(L, τ') > 0 for every row of X_feats
    mask = np.ones(len(X_feats), dtype=bool)
    for j, e, s in conjuncts:
        mask &= e * X_feats[:, j] > s
    return mask

In [62]:
_raw_stats = {}

def raw_stats(j):
    # Train mean / std of ρ(ϕ_j, ·), as used for standardization in build_formula_bank
    if j not in _raw_stats:
        rho = eval_robustness(formulas[j], X_tr_raw)
        _raw_stats[j] = (float(rho.mean()), float(rho.std()))
    return _raw_stats[j]


def reparametrize(j, e, s):
    # Eq. (21): ϕ̃_{j,c} = (ϕ_j)_{-c} or (¬ϕ_j)_{-c}, with c = ε_j μ_j + s (σ_j + 1e-8)
    mu, sigma = raw_stats(j)
    c = e * mu + s * (sigma + 1e-8)
    phi = copy.deepcopy(formulas[j])
    if e < 0:
        phi = Not(phi)
    shift_atom_thresholds(phi, -c)
    return phi


def to_stl(conjuncts):
    return reduce(And, [reparametrize(*cj) for cj in conjuncts]) if conjuncts else None


# Check: the torcheck robustness of a reparametrized explanation reproduces the
# feature-space covered set (up to float ties at the cut, where ρ ≈ 0)
expl = explain_local(X_te_feats[0])
L = to_stl(expl["conjuncts"])
rho_L = eval_robustness(L, X_tr_raw)
mismatch = (rho_L > 0) != covered(expl["conjuncts"], X_tr_feats)
print(f"reparametrization check: {mismatch.sum()} / {len(mismatch)} mismatches")
assert not mismatch.any() or np.abs(rho_L[mismatch]).max() < 1e-4 * np.abs(rho_L).max()

reparametrization check: 0 / 160 mismatches


## Local explanations

Every test series gets a local explanation from the beam search, and a **baseline** with the same number of conjuncts: the top-IG formulae of $J$ in IG order, each with the best cut from `candidate_cuts` given the previous ones. The baseline optimizes the cuts but not the choice of formulae.

Metrics, w.r.t. the model predictions $f$, on the **train** set (in-sample: the search optimized there) and the **test** set (held-out, explained series excluded):

- **pooled prec**: $\sum\mathrm{TP}/\sum|\mathcal C|$ over explanations, with a 95% bootstrap CI over explained series; **prec>α**: share of explanations whose own precision exceeds α.
- **in-class cov**: $\mathrm{TP}/|\{\tau' : f(\tau')=k\}|$, the share of series predicted as $k$ that the explanation covers.
- **length**: number of conjuncts. **feasible**: share of explanations with train precision $>\alpha$.

In [63]:
def baseline_local(x_hat, n, alpha=ALPHA):
    # Top-n formulae of J by IG, each with the best cut given the previous conjuncts
    k, ig, d, _ = integrated_gradients(x_hat)
    J = np.flatnonzero(ig > 0)
    pos, cov, conjuncts = f_tr == k, np.ones(len(f_tr), dtype=bool), []
    for j in J[np.argsort(-ig[J])[:n]]:
        e = float(np.sign(d[j]))
        u = e * X_tr_feats[:, j]
        s_G, s_F = candidate_cuts(u, e * x_hat[j], cov, pos, alpha)
        s = s_F if s_F is not None else s_G
        cov &= u > s
        conjuncts.append((int(j), e, float(s)))
    return dict(k=k, conjuncts=conjuncts, feasible=bool((cov & pos).sum() > alpha * cov.sum()))


def local_table(expl, method):
    # One row per explained test series: in-sample (train) and held-out (test, self excluded)
    rows = []
    for i, e in enumerate(expl):
        if not e["conjuncts"]:
            continue
        row = dict(method=method, length=len(e["conjuncts"]), feasible=e["feasible"])
        for split, X_feats, f in (("train", X_tr_feats, f_tr), ("test", X_te_feats, f_te)):
            mask, pos = covered(e["conjuncts"], X_feats), f == e["k"]
            if split == "test":
                mask[i] = pos[i] = False
            row |= {f"tp_{split}": int((mask & pos).sum()), f"cov_{split}": int(mask.sum()),
                    f"prec_{split}": (mask & pos).sum() / mask.sum() if mask.any() else np.nan,
                    f"recall_{split}": (mask & pos).sum() / pos.sum() if pos.any() else np.nan}
        rows.append(row)
    return pd.DataFrame(rows)


def summarize(df, n_boot=1000, seed=0):
    rng = np.random.default_rng(seed)
    rows = []
    for method, g in df.groupby("method", sort=False):
        for split in ("train", "test"):
            tp, cov = g[f"tp_{split}"].to_numpy(), g[f"cov_{split}"].to_numpy()
            idx = rng.integers(0, len(g), size=(n_boot, len(g)))
            boot = tp[idx].sum(1) / np.maximum(cov[idx].sum(1), 1)
            rows.append({"method": method, "split": split, "n": len(g),
                         "pooled prec": tp.sum() / cov.sum(),
                         "95% CI": f"[{np.quantile(boot, .025):.3f}, {np.quantile(boot, .975):.3f}]",
                         "prec>α": (g[f"prec_{split}"] > ALPHA).mean(),
                         "in-class cov": g[f"recall_{split}"].mean(),
                         "length": g["length"].mean(),
                         "feasible": g["feasible"].mean()})
    return pd.DataFrame(rows).set_index(["method", "split"]).round(3)

In [64]:
test_expl = [explain_local(X_te_feats[i]) for i in range(len(X_te_feats))]
base_expl = [baseline_local(X_te_feats[i], len(e["conjuncts"])) for i, e in enumerate(test_expl)]

# Consistency on the actual STL formulas: ρ(L(τ), τ) > 0 for every explained test series
rho_self = np.array([eval_robustness(to_stl(e["conjuncts"]), X_te_raw[i:i+1])[0]
                     for i, e in enumerate(test_expl) if e["conjuncts"]])
print(f"without explanation: {sum(not e['conjuncts'] for e in test_expl)} / {len(test_expl)}, "
      f"consistency: {(rho_self > 0).mean():.3f} (min ρ = {rho_self.min():.2e})")

summarize(pd.concat([local_table(test_expl, "beam"), local_table(base_expl, "baseline")]))

without explanation: 0 / 74, consistency: 1.000 (min ρ = 3.25e-02)


n  pooled prec          95% CI  prec>α  in-class cov  length  \
method   split                                                                  
beam     train  74        0.952  [0.943, 0.961]   1.000         0.237   2.473   
         test   74        0.701  [0.657, 0.743]   0.243         0.143   2.473   
baseline train  74        0.946  [0.915, 0.970]   0.757         0.068   2.473   
         test   74        0.805  [0.731, 0.869]   0.365         0.051   2.473   

                feasible  
method   split            
beam     train     1.000  
         test      1.000  
baseline train     0.757  
         test      0.757

## Global explanations

$\mathcal G(k)=\bigvee_{i\in\mathcal I_{\rm train}: f(\tau^{(i)})=k}\mathcal L(\tau^{(i)})$ over the unique train-set local explanations. A disjunct is dropped when another disjunct is implied by it (every conjunct of the other has a conjunct on the same $(j,\epsilon_j)$ with a cut at least as large), which leaves the formula logically equivalent (**exact**).

The **simplified** version then applies the library simplification (paper §3.4), each step keeping the train covered set ≥ `simplify_agreement` identical: `simplify_global` merges disjuncts with near-identical covered sets and drops those that do not improve coverage − FP rate (w.r.t. the model predictions on train, not the true labels as in the library pipeline), `simplify_data_aware` removes subformulae, and `round_thresholds` rounds thresholds to `simplify_decimals` decimals. It is approximate: behaviour on unseen series can change.

In [65]:
def implies(b, a):
    # b ⇒ a for every series: each conjunct (j, ε, s) of a has a conjunct of b
    # on the same oriented formula with a cut at least as tight
    tight = {}
    for j, e, s in b:
        tight[(j, e)] = max(s, tight.get((j, e), -np.inf))
    return all(tight.get((j, e), -np.inf) >= s for j, e, s in a)


def remove_subsumed(disjuncts):
    # Drop every disjunct implied by another one (keep one copy of equivalent ones)
    return [b for i, b in enumerate(disjuncts)
            if not any(implies(b, a) and (not implies(a, b) or l < i)
                       for l, a in enumerate(disjuncts) if l != i)]


train_expl = [explain_local(X_tr_feats[i]) for i in range(len(X_tr_feats))]
locals_by_class = defaultdict(dict)
for e in train_expl:
    if e["conjuncts"]:
        locals_by_class[e["k"]][tuple(e["conjuncts"])] = e["conjuncts"]

def n_atoms(phi):
    if isinstance(phi, Atom):
        return 1
    return sum(n_atoms(getattr(phi, a)) for a in ("child", "left_child", "right_child")
               if getattr(phi, a, None) is not None)


agree = config.simplify_agreement
global_per_class, global_simplified, size = {}, {}, {}
for k, D in sorted(locals_by_class.items()):
    cls = model.classes_[k]
    phis = [to_stl(c) for c in remove_subsumed(list(D.values()))]
    global_per_class[cls] = reduce(Or, phis)

    kept = simplify_global(phis, X_tr_raw, model.classes_[f_tr], cls, agree, config.simplify_min_gain)
    phi = simplify_data_aware(reduce(Or, kept), X_tr_raw, agree)
    global_simplified[cls] = round_thresholds(phi, X_tr_raw, config.simplify_decimals, agree)

    size[cls] = {"disjuncts exact": len(phis), "atoms exact": n_atoms(global_per_class[cls]),
                 "disjuncts simplified": len(kept), "atoms simplified": n_atoms(global_simplified[cls])}
    print(f"{cls!s:<6} {global_simplified[cls]}")

pd.DataFrame(size).T

backward ( ( ( ( ( ( ¬( F[137,260](x_6 <= -278.6) ) ) ∧ ( ¬( G[109,209]( (x_4 <= 304.2) ∨ (x_1 => 217.4) ) ) ) ) ∨ ( ( ( ¬( G[172,225]( G[12,16](x_2 <= 333.3) ) ) ) ∧ ( ¬( F[170,272](x_4 <= -286.7) ) ) ) ∧ ( ¬( G[219,225](x_2 <= 101.8) ) ) ) ) ∨ ( ( ¬( G[219,225](x_2 <= 195.2) ) ) ∧ ( ¬( F[137,260](x_6 <= -329.4) ) ) ) ) ∨ ( ( F[275,288]( F[15,18](x_4 <= -288.9) ) ) ∧ ( G[213,257](x_8 <= 208.8) ) ) ) ∨ ( ( ¬( F[137,260](x_6 <= -253.9) ) ) ∧ ( F[275,288]( F[15,18](x_4 <= -153.7) ) ) ) )
forward ( ( ( ( ( ( ( F[170,272](x_4 <= -358.8) ) ∧ ( ¬( F[200,inf]( G[115,182](x_9 <= 221.9) ) ) ) ) ∨ ( ( ( ¬( G[322,361](x_6 <= 296.1) ) ) ∧ ( G[219,225](x_2 <= 67.5) ) ) ∧ ( ¬( F[200,inf]( G[115,182](x_9 <= 219.2) ) ) ) ) ) ∨ ( ( ¬( F[244,264](x_1 => 25.6) ) ) ∧ ( ¬( F[200,inf]( G[115,182](x_9 <= 222.0) ) ) ) ) ) ∨ ( ( F[164,inf]( (x_3 => 258.7) ∧ (x_1 <= 24.1) ) ) ∧ ( ¬( G[318,340](x_3 <= 191.4) ) ) ) ) ∨ ( ( ¬( F[292,326](x_8 <= -150.0) ) ) ∧ ( F[245,290](x_7 <= -337.6) ) ) ) ∨ ( ( ¬( F[200,inf]( G

,disjuncts exact,atoms exact,disjuncts simplified,atoms simplified
backward,31,104,9,12
forward,36,88,9,14
left,24,86,7,8
right,20,67,3,6


### Tuning the global explanations
Each simplified $\mathcal G(k)$ is fine-tuned by `stlrocket.tuning.finetune_formula`: atom thresholds and bounded time intervals are refined by gradient descent on a smooth F1 surrogate, w.r.t. the **model predictions** on train (fidelity). The tuned formula is kept only if its exact train F1 does not decrease; that gate is in-sample, so the test columns below are the real check.

In [66]:
from stlrocket.tuning import finetune_formula

global_tuned, tune_log = {}, {}
for cls, phi in global_simplified.items():
    global_tuned[cls], log = finetune_formula(phi, X_tr_raw, model.classes_[f_tr], cls)
    tune_log[cls] = {"train F1 before": log["f1_before"], "train F1 after": log["f1_after"],
                     "accepted": log["accepted"]}
    print(f"{cls!s:<6} {global_tuned[cls]}")

pd.DataFrame(tune_log).T

backward ( ( ( ( ( ( ¬( F[137,260](x_6 <= -278.6) ) ) ∧ ( ¬( G[109,209]( (x_4 <= 304.2) ∨ (x_1 => 217.4) ) ) ) ) ∨ ( ( ( ¬( G[172,225]( G[12,16](x_2 <= 333.3) ) ) ) ∧ ( ¬( F[170,272](x_4 <= -286.7) ) ) ) ∧ ( ¬( G[219,225](x_2 <= 101.8) ) ) ) ) ∨ ( ( ¬( G[219,225](x_2 <= 195.2) ) ) ∧ ( ¬( F[137,260](x_6 <= -329.4) ) ) ) ) ∨ ( ( F[275,288]( F[15,18](x_4 <= -288.9) ) ) ∧ ( G[213,257](x_8 <= 208.8) ) ) ) ∨ ( ( ¬( F[137,260](x_6 <= -253.9) ) ) ∧ ( F[275,288]( F[15,18](x_4 <= -153.7) ) ) ) )
forward ( ( ( ( ( ( ( F[170,272](x_4 <= -358.8) ) ∧ ( ¬( F[200,inf]( G[115,182](x_9 <= 221.9) ) ) ) ) ∨ ( ( ( ¬( G[322,361](x_6 <= 296.1) ) ) ∧ ( G[219,225](x_2 <= 67.5) ) ) ∧ ( ¬( F[200,inf]( G[115,182](x_9 <= 219.2) ) ) ) ) ) ∨ ( ( ¬( F[244,264](x_1 => 25.6) ) ) ∧ ( ¬( F[200,inf]( G[115,182](x_9 <= 222.0) ) ) ) ) ) ∨ ( ( F[164,inf]( (x_3 => 258.7) ∧ (x_1 <= 24.1) ) ) ∧ ( ¬( G[318,340](x_3 <= 191.4) ) ) ) ) ∨ ( ( ¬( F[292,326](x_8 <= -150.0) ) ) ∧ ( F[245,290](x_7 <= -337.6) ) ) ) ∨ ( ( ¬( F[200,inf]( G

,train F1 before,train F1 after,accepted
backward,0.851064,0.53913,False
forward,0.897196,0.666667,False
left,0.827586,0.537313,False
right,0.65,0.411765,False


In [67]:
# Test-set F1 of G(k) w.r.t. the true labels and w.r.t. the model predictions (fidelity)
res = {f"F1 {target} ({name})": evaluate_global(G, X_te_raw, y)
       for name, G in (("exact", global_per_class), ("simplified", global_simplified),
                       ("tuned", global_tuned))
       for target, y in (("labels", y_te), ("model", model.classes_[f_te]))}
pd.DataFrame({name: {cls: r[cls]["f1"] for cls in r} for name, r in res.items()}).round(3)

,F1 labels (exact),F1 model (exact),F1 labels (simplified),F1 model (simplified),F1 labels (tuned),F1 model (tuned)
backward,0.419,0.605,0.387,0.710,0.387,0.710
forward,0.512,0.736,0.400,0.738,0.400,0.738
left,0.606,0.545,0.500,0.333,0.500,0.333
right,0.343,0.467,0.167,0.526,0.167,0.526
macro_avg,0.470,0.588,0.363,0.577,0.363,0.577
